# Ejercicio 5 - Incorporacion de datos de 2024

## 5.1-5.4 Descarga
```bash
python scripts/download_data.py --anios 2026 2024   # (2024 ya forma parte de ANIOS por defecto)
```
Primera ejecucion: 23 archivos descargados (todos menos `yellow 2024-03`, que un `HEAD` fallido marco como "no publicado"),
16 de 2026 omitidos por existir. Ese fallo transitorio llevo a corregir `esta_publicado` para reintentar y solo declarar
"no publicado" ante 403/404. Segunda ejecucion: 1 descargado (`yellow 2024-03`). Tercera: 0 descargados, 40 existentes.

In [1]:
import sys; sys.path.insert(0, "../scripts")
import labutil as lu
con = lu.connect(anio="*")   # todos los anios presentes en data/raw

## 5.5 Archivos incorporados
Verificacion adicional por linea de comandos: `python scripts/verify_data.py --anios 2024 2026`.

In [2]:
lu.q(con, "05_01_files_by_year.sql")

-- 05_01_files_by_year.sql
-- Objetivo: confirmar que los archivos de 2024 y 2026 conviven en el glob (5.5, 5.6).
-- Fuente: data/raw/*/*/*.parquet
SELECT regexp_extract(file, '/(yellow|green)/', 1) AS taxi,
       regexp_extract(file, '/(\d{4})/', 1) AS anio, count(*) AS archivos
FROM glob('/Users/a87009/Documents/Año 4 Semestre 2/Data Science/Laboratorio 8 DuckDB/lab8-duckdb/data/raw/*/*/*.parquet') AS t(file) GROUP BY ALL ORDER BY taxi, anio;



,taxi,anio,archivos
0,green,2024,12
1,green,2026,8
2,yellow,2024,12
3,yellow,2026,8


**Resultado:** 12 archivos por taxi para 2024 y 8 por taxi para 2026 (40 en total).

## 5.6 Consulta conjunta de 2024 y 2026

In [3]:
lu.q(con, "05_02_rows_by_year.sql")

-- 05_02_rows_by_year.sql
-- Objetivo: consulta conjunta de 2024 y 2026 sin cambiar la vista (5.6).
-- Fuente: vista trips (Parquet de todos los anios).
SELECT data_year AS anio, taxi, count(*) AS viajes, min(pickup) AS primer_pickup, max(pickup) AS ultimo_pickup
FROM trips GROUP BY ALL ORDER BY anio, taxi;



,anio,taxi,viajes,primer_pickup,ultimo_pickup
0,2024,green,660218,2008-12-31 00:00:00,2025-01-01 22:21:15
1,2024,yellow,41169720,2002-12-31 16:46:07,2026-06-26 23:53:12
2,2026,green,337114,2008-12-31 17:35:31,2026-08-31 23:58:28
3,2026,yellow,29703355,2001-01-01 09:23:58,2026-08-31 23:59:59


**Resultado:** el mismo `trips` devuelve ambos anios sin ningun cambio en la vista. Los `min/max(pickup)` muestran de nuevo fechas
erroneas (2001-2009 y hasta 2026 dentro de archivos 2024), que confirman la necesidad de `trips_clean`.

## 5.7 Las consultas anteriores necesitan modificarse?

**No.** Las vistas leen `data/raw/<tipo>/*/*.parquet` con `union_by_name`, y las consultas agrupan por `data_year/data_month`
o no dependen del anio. Se comprueba a continuacion el unico riesgo real: columnas que cambian entre anios (*schema drift*).

In [4]:
lu.q(con, "05_03_schema_drift.sql")

-- 05_03_schema_drift.sql
-- Objetivo: detectar columnas que cambian entre anios (5.7).
-- Fuente: vista trips. Cuenta valores no nulos de columnas introducidas despues de 2024.
SELECT data_year AS anio, taxi, count(*) AS filas,
       count(cbd_congestion_fee) AS con_cbd_fee,
       count(airport_fee)        AS con_airport_fee,
       count(passenger_count)    AS con_pasajeros,
       count(*) FILTER (WHERE payment_type = 0) AS pago_tipo_0
FROM trips GROUP BY ALL ORDER BY anio, taxi;



,anio,taxi,filas,con_cbd_fee,con_airport_fee,con_pasajeros,pago_tipo_0
0,2024,green,660218,0,0,635890,0
1,2024,yellow,41169720,0,37078488,37078488,4091232
2,2026,green,337114,337114,0,288339,0
3,2026,yellow,29703355,29703355,21986667,21986667,7716688


**Resultado:**
- `cbd_congestion_fee` (cargo por congestion de Manhattan, vigente desde 2025) es **NULL en todo 2024**; en 2026 siempre esta presente.
  `union_by_name` lo rellena con NULL, asi que no rompe nada; cualquier calculo que lo sume debe usar `coalesce`.
- `airport_fee` existe en yellow para ambos anios.
- `payment_type = 0` y los nulos en pasajeros casi no aparecen en 2024 (4.1 M de 41 M en yellow, 10%) frente a 2026 (26%): cambio de captura
  detectado ya en el EDA, y ahora visible como tendencia.
- Los tipos de columna (`passenger_count` entero o decimal segun el anio) se unifican con `CAST` en la vista.
- **Unico cambio necesario:** al consultar *solo* 2024 (Ejercicio 6), la vista fallaba porque `cbd_congestion_fee` no existe en esos archivos. Se agrego un esquema "ancla" (`UNION ALL BY NAME ... WHERE false`) en `00_views.sql` que declara las columnas con deriva; el resto de consultas queda igual.

In [5]:
lu.q(con, "05_04_monthly_both_years.sql").pivot(index=["anio", "mes"], columns="taxi", values="viajes")

-- 05_04_monthly_both_years.sql
-- Objetivo: la consulta mensual de 04_01 reutilizada tal cual con dos anios (5.7).
-- Fuente: trips_clean.
SELECT data_year AS anio, data_month AS mes, taxi, count(*) AS viajes
FROM trips_clean GROUP BY ALL ORDER BY anio, mes, taxi;



taxi      green   yellow
anio mes                
2024 1    53302  2867786
     2    50394  2899629
     3    54094  3437844
     4    52931  3412000
     5    57455  3614191
     6    51671  3425962
     7    48484  2971888
     8    48658  2865358
     9    51241  3481885
     10   53155  3679813
     11   49065  3507838
     12   50580  3516591
2026 1    38217  3515618
     2    35238  3209117
     3    41797  3761390
     4    41714  3672810
     5    42475  3910683
     6    41678  3644819
     7    38699  3344952
     8    38169  3162470

**Resultado:** la consulta mensual de `04_01_monthly.sql` se reutiliza con el mismo texto (solo se amplia su resultado).

## 5.8 Consultas de validacion
`sql/05_01_files_by_year.sql`, `05_02_rows_by_year.sql`, `05_03_schema_drift.sql` y `05_04_monthly_both_years.sql`, impresas arriba.

## 5.9 Que permite incorporar archivos nuevos sin rehacer el flujo
- **Datos separados del codigo y organizados por convencion:** `data/raw/<tipo>/<anio>/<archivo>.parquet`; el glob de las vistas es la unica puerta de entrada.
- **Vistas en lugar de copias:** agregar un archivo lo hace visible de inmediato; no hay etapa de carga.
- **`union_by_name`** tolera columnas nuevas/ausentes entre anios; **un esquema unificado** (`trips`) aisla al analisis de los nombres originales.
- **Calidad centralizada en `trips_clean`:** una sola definicion de "viaje valido".
- **Descarga idempotente** y parametrizada por anio: repetirla es seguro y solo trae lo que falta.